# 03 · Năm mô hình và đối chứng
Logistic Regression, Decision Tree, Random Forest, KNN, SVM + Dummy. Tìm siêu tham số trên cùng 5 fold theo nhóm. Chọn mô hình theo train CV AP; chọn ngưỡng bằng validation F2. Bài giảng: trang 46, 55–57, 61, 66.

CV sau tìm kiếm có thiên lệch lạc quan; đây không phải nested CV. Test độc lập dùng báo cáo cuối.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (ROOT / 'src').exists(), 'Open notebook from project or notebooks directory'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.data_engine import PROFILE_FILE, COURSE_FILE, MODEL_DIR, FEATURES
profiles = pd.read_csv(PROFILE_FILE)
records = pd.read_csv(COURSE_FILE)


In [ ]:
from src.experiments import candidates
for name, (_, grid) in candidates().items():
    print(name, grid)

In [ ]:
# Đổi True để huấn luyện lại. Mặc định đọc kết quả đã có, tránh vô tình ghi artifact.
RUN_TRAINING = False
if RUN_TRAINING:
    from src.training import train
    metrics = train(profiles, records, folds_count=5, jobs=1)
else:
    assert (MODEL_DIR / 'metrics.csv').exists(), 'Run python -m src.training first'
    metrics = pd.read_csv(MODEL_DIR / 'metrics.csv')
display(metrics[['Model', 'CV_AP_mean', 'CV_AP_std', 'Best_Params', 'Threshold', 'Selected']])

## Ngưỡng
Tối đa F2 trên validation trong lưới 0.05–0.95, bước 0.01; hòa chọn Precision cao hơn rồi ngưỡng cao hơn. Không refit trên validation sau khi chốt ngưỡng. Không chọn mô hình từ điểm test. Các bản chạy được giữ trong results/academic_model/runs; artifact trước được sao lưu trong archive.